In [2]:
import math
import random

# ── Dados ─────────────────────────────────────────────────────────────────────

texts = [
    # saudação
    "oi", "olá", "boa tarde", "bom dia", "boa noite",
    "oi tudo bem", "olá como vai", "ei boa tarde",
    "oi preciso de ajuda", "olá estou aqui",

    # reclamação
    "não funciona", "péssimo serviço", "estou com problema",
    "não consigo acessar", "travou de novo", "que absurdo",
    "meu pedido não chegou", "horrível isso aqui",
    "não tô conseguindo", "sistema caiu",

    # dúvida
    "como faço para", "qual o prazo", "onde fica",
    "quanto custa", "quais são os horários", "como cancelo",
    "preciso saber o status", "tem como verificar",
    "me explica como funciona", "o que é isso",

    # elogio
    "muito bom", "adorei", "excelente atendimento",
    "parabéns pelo serviço", "ótimo", "super rápido",
    "vocês são incríveis", "amei a experiência",
    "perfeito obrigado", "muito satisfeito",
]

labels = [
    "saudacao","saudacao","saudacao","saudacao","saudacao",
    "saudacao","saudacao","saudacao","saudacao","saudacao",

    "reclamacao","reclamacao","reclamacao","reclamacao","reclamacao",
    "reclamacao","reclamacao","reclamacao","reclamacao","reclamacao",

    "duvida","duvida","duvida","duvida","duvida",
    "duvida","duvida","duvida","duvida","duvida",

    "elogio","elogio","elogio","elogio","elogio",
    "elogio","elogio","elogio","elogio","elogio",
]

# ── TF-IDF do zero ────────────────────────────────────────────────────────────

def tokenize(text):
    return text.lower().split()

def build_vocab(docs):
    vocab = {}
    for doc in docs:
        for token in tokenize(doc):
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

def fit_transform(docs, vocab):
    N, V = len(docs), len(vocab)
    df = [0] * V
    tfs = []
    for doc in docs:
        tokens = tokenize(doc)
        tf = [0.0] * V
        for t in tokens:
            if t in vocab:
                tf[vocab[t]] += 1 / len(tokens)
        tfs.append(tf)
        for i, v in enumerate(tf):
            if v > 0:
                df[i] += 1

    idf = [math.log((N + 1) / (df[i] + 1)) + 1 for i in range(V)]

    vecs = []
    for tf in tfs:
        v = [tf[i] * idf[i] for i in range(V)]
        norm = math.sqrt(sum(x**2 for x in v))
        vecs.append([x / norm for x in v] if norm > 0 else v)
    return vecs, idf

def transform(text, vocab, idf):
    V = len(vocab)
    tokens = tokenize(text)
    tf = [0.0] * V
    for t in tokens:
        if t in vocab:
            tf[vocab[t]] += 1 / len(tokens)
    v = [tf[i] * idf[i] for i in range(V)]
    norm = math.sqrt(sum(x**2 for x in v))
    return [x / norm for x in v] if norm > 0 else v

# ── Perceptron multiclasse ────────────────────────────────────────────────────

def dot(a, b):
    return sum(a[i] * b[i] for i in range(len(a)))

def train(vecs, labels, epochs=300):
    classes = list(set(labels))
    V = len(vecs[0])
    weights = {c: [0.0] * V for c in classes}

    def predict(x):
        return max(classes, key=lambda c: dot(weights[c], x))

    for _ in range(epochs):
        # embaralha a cada época — melhora a convergência
        pairs = list(zip(vecs, labels))
        random.shuffle(pairs)
        for x, y in pairs:
            pred = predict(x)
            if pred != y:
                for i in range(V):
                    weights[y][i]    += x[i]
                    weights[pred][i] -= x[i]

    return weights, classes, predict

# ── Treino ────────────────────────────────────────────────────────────────────

vocab = build_vocab(texts)
vecs, idf = fit_transform(texts, vocab)
weights, classes, _predict = train(vecs, labels)

def classify(text):
    v = transform(text, vocab, idf)
    scores = {c: dot(weights[c], v) for c in classes}
    best = max(scores, key=scores.get)
    return best, scores

# ── Teste ─────────────────────────────────────────────────────────────────────

testes = [
    "oi boa tarde",
    "não consigo entrar no sistema",
    "como faço para cancelar",
    "adorei o atendimento",
    "que droga isso não funciona",
    "qual o horário de funcionamento",
]

for texto in testes:
    intencao, scores = classify(texto)
    scores_fmt = " | ".join(f"{c}: {v:+.2f}" for c, v in sorted(scores.items()))
    print(f'"{texto}"')
    print(f"  → {intencao}  ({scores_fmt})\n")

"oi boa tarde"
  → saudacao  (duvida: -0.26 | elogio: +0.00 | reclamacao: -0.92 | saudacao: +1.18)

"não consigo entrar no sistema"
  → reclamacao  (duvida: -0.31 | elogio: +0.00 | reclamacao: +0.31 | saudacao: +0.00)

"como faço para cancelar"
  → duvida  (duvida: +0.18 | elogio: +0.00 | reclamacao: -0.18 | saudacao: +0.00)

"adorei o atendimento"
  → elogio  (duvida: +0.22 | elogio: +1.04 | reclamacao: -1.27 | saudacao: +0.00)

"que droga isso não funciona"
  → reclamacao  (duvida: -0.43 | elogio: +0.00 | reclamacao: +0.71 | saudacao: -0.29)

"qual o horário de funcionamento"
  → reclamacao  (duvida: -0.05 | elogio: +0.00 | reclamacao: +0.07 | saudacao: -0.02)

